In [16]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import pystormtracker as pst
from pathlib import Path
import cartopy.crs as ccrs

import sys
import os

module_dir = (Path.cwd() / ".."/ ".." / "algorithms").resolve()
sys.path.insert(0,str(module_dir))
import validation as val
import processing as pro
import helpers
from configs import *

import importlib
importlib.reload(val)  # pick up edits in validation.py without restarting the kernel
importlib.reload(pro)  # pick up edits in processing.py without restarting the kernel



<module 'processing' from '/Users/asgerthormann/Documents/uni/master/master_code/master_thesis/algorithms/processing.py'>

In [17]:
def share_with_partner(a, b, max_km=100):
    a = a[["time", "lat", "lon"]].reset_index(drop=True)
    a["row"] = np.arange(len(a))                        # explicit point id, independent of the index name
    m = a.merge(b[["time", "lat", "lon"]], on="time", suffixes=("_a", "_b"))
    m["d"] = helpers.haversine_km(m["lat_a"], m["lon_a"], m["lat_b"], m["lon_b"])
    nearest = m.groupby("row")["d"].min().reindex(a["row"])   # NaN where no point of b exists at that time
    return (nearest <= max_km).mean()



In [18]:
NH_BOX = (-180, 180, 0, 90)                    # lon_min, lon_max, lat_min, lat_max
NH_CONFIG = {**POSTPROCESS_CONFIG, "region": NH_BOX, "domain": NH_BOX,
             "max_frac_at_edge": 1.0}          # no real edges: the equator and ±180° aren't boundaries


In [19]:
TIME_RANGE = ('1998-12-01 00:00:00', '1999-02-28 18:00:00')  # time range for the validation tracks
df_validation = val.load_tracks('/Users/asgerthormann/Documents/uni/master/master_code/master_thesis/validation_tracks_data/ERA5_19981999_TRACKS_all_pos.addmslp', start_time=TIME_RANGE[0], end_time=TIME_RANGE[1])
df_validation=df_validation.rename(columns={'vorticity': 'vo'})

In [20]:
df_validation
len(df_validation['track_id'].unique())

2492

,step,lon,lat,vo,mslp_lon,mslp_lat,mslp,track_id,time
obs,,,,,,,,,
0,1,-8.480621,29.504873,3.483846,351.8229,29.26017,1007.5360,1,1998-12-01 00:00:00
1,2,-7.940887,28.378540,3.742608,352.5893,28.43965,1008.2640,1,1998-12-01 06:00:00
2,3,-6.786499,28.493826,4.328402,355.7565,29.30679,1008.9330,1,1998-12-01 12:00:00
3,4,-4.055176,28.891768,4.159280,356.1487,29.71470,1006.8400,1,1998-12-01 18:00:00
4,5,-1.789795,29.933268,3.795781,358.6745,29.89172,1008.6970,1,1998-12-02 00:00:00
...,...,...,...,...,...,...,...,...,...
19013,351,-142.131470,45.803730,6.270992,NaN,NaN,988.0286,2580,1999-02-26 12:00:00
19014,352,-138.802780,47.091911,7.628706,218.0783,48.09399,981.6581,2580,1999-02-26 18:00:00
19015,353,-136.685135,49.011963,8.372078,222.5054,48.51379,977.2754,2580,1999-02-27 00:00:00


In [21]:
tracks = pst.load_tracks('/Users/asgerthormann/Documents/uni/master/master_code/offline_backup/trackjson/19981999_DJF_vo_lmin_6.trackjson')
tracks.metadata.processing

(ProcessingStep(operation='spectral_filter', enabled=True, parameters=mappingproxy({'method': 'sht', 'lmin': 6, 'lmax': 42, 'spectral_taper': 0.1})),)

In [22]:
df = helpers.tracks_to_points(tracks)
len(df['track_id'].unique())

3705

In [23]:
df = val.clip_to_domain(df, NH_BOX, TIME_RANGE)   # instead of df[df.lat > 0]: also gives pieces their own ids
len(df['track_id'].unique())

1766

In [24]:
print(f"his points with one of yours within 100 km: {share_with_partner(df_validation, df):.3f}")
print(f"your points with one of his within 100 km:  {share_with_partner(df, df_validation):.3f}")

his points with one of yours within 100 km: 0.893
your points with one of his within 100 km:  0.946


In [29]:
tracks = val.dataframe_to_tracks(df, variable = 'vo', scale=1)  # already in s^-1, so scale=1

In [25]:
tracks_validation = val.dataframe_to_tracks(df_validation, variable = 'vo', scale=1e-5)  # convert to s^-1

count    2492.000000
mean        7.631621
std        10.523012
min         1.000000
25%         2.000000
50%         4.000000
75%         9.000000
max       292.000000
dtype: float64
his points with one of yours within 100 km: 0.940
your points with one of his within 100 km:  0.931
0.8855664174813112 0.8881199538638985 0.8868413475381515


In [30]:
comparison, matches_df = val.compare_to_validation(tracks_validation, tracks, matching="global_assignment",
                                               max_mean_separation_deg=2.0, min_overlap_fraction=0.6,
                                               variable="vo")

In [31]:
matches_df

,reference_id,candidate_id,overlap_fraction,overlap_count,mean_sep_km,max_sep_km,ref_points,cand_points,ref_peak,cand_peak,n_eligible_candidates,intensity_bias,intensity_rmse
0,1,2900,1.000000,5,9.202616,16.655725,5,5,0.000043,0.000044,1,2.195138e-07,2.581201e-07
1,2,2800,1.000000,7,11.407177,19.944699,7,7,0.000045,0.000045,1,2.637139e-07,2.874351e-07
2,3,13300,0.857143,6,10.850059,16.757137,8,6,0.000046,0.000046,1,-1.380136e-08,9.914685e-08
3,4,13700,0.888889,8,15.875251,39.429584,10,8,0.000027,0.000026,1,2.483227e-08,5.279461e-08
4,5,4000,1.000000,28,14.903127,30.540707,28,28,0.000032,0.000032,1,1.129568e-07,1.734946e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,2565,620600,1.000000,5,18.431287,58.702144,5,5,0.000016,0.000016,1,9.776728e-08,2.031205e-07
1618,2566,620700,0.947368,18,12.503462,25.272740,19,19,0.000021,0.000021,1,7.504113e-08,1.150797e-07
1619,2569,600400,0.909091,5,22.088519,39.981908,5,6,0.000019,0.000019,1,-1.229005e-09,7.498336e-08
1620,2575,532700,1.000000,9,16.658837,27.689117,9,9,0.000026,0.000026,1,5.056013e-08,7.583956e-08


In [32]:
# result table 
results_df = pd.Series({
    "Hits": comparison.tp,
    "Miss": comparison.fn,
    "Extra": comparison.fp,
    "F1": comparison.f1, #
    "Precision": comparison.precision,
    "Recall": comparison.recall,
})
results_df

Hits         1622.000000
Miss          870.000000
Extra         144.000000
F1              0.761860
Precision       0.918460
Recall          0.650883
dtype: float64

In [35]:
dv = val.clip_to_domain(df_validation, NH_BOX, TIME_RANGE)          # his raw, same clipping as yours
print(dv.groupby("track_id").size().describe())                     # how short are his shortest raw tracks?

min_len = 3                                                         # your min_track_points; use his minimum if larger
long_enough = lambda d: d[d.groupby("track_id")["track_id"].transform("size") >= min_len]
dv, dm = long_enough(dv), long_enough(df)

print(f"his points with one of yours within 100 km: {share_with_partner(dv, dm):.3f}")
print(f"your points with one of his within 100 km:  {share_with_partner(dm, dv):.3f}")

ref_raw = val.dataframe_to_tracks(dv, variable="vo", scale=1e-5)
exp_raw = val.dataframe_to_tracks(dm, variable="vo", scale=1.0)
comparison, matches_df = val.compare_to_validation(ref_raw, exp_raw, "global_assignment",
                                                   2.0, 0.6, variable="vo")
print(comparison.recall, comparison.precision, comparison.f1)


count    2492.000000
mean        7.631621
std        10.523012
min         1.000000
25%         2.000000
50%         4.000000
75%         9.000000
max       292.000000
dtype: float64
his points with one of yours within 100 km: 0.940
your points with one of his within 100 km:  0.931
0.8855664174813112 0.8881199538638985 0.8868413475381515


In [36]:
def describe_pairing(a, b, scale_a, max_km=100):
    a = a.reset_index(drop=True).copy()
    a["row"] = np.arange(len(a))
    m = a[["row", "time", "lat", "lon"]].merge(b[["time", "lat", "lon"]], on="time", suffixes=("", "_b"))
    m["d"] = helpers.haversine_km(m["lat"], m["lon"], m["lat_b"], m["lon_b"])
    a["nearest_km"] = m.groupby("row")["d"].min().reindex(a["row"]).to_numpy()
    a["paired"] = a["nearest_km"] <= max_km
    a["vo_1e5"] = a["vo"] * scale_a * 1e5                                  # both in 1e-5 s^-1
    a["track_len"] = a.groupby("track_id")["track_id"].transform("size")
    return a

his  = describe_pairing(dv, dm, scale_a=1e-5)
mine = describe_pairing(dm, dv, scale_a=1.0)

for name, d in [("his", his), ("yours", mine)]:
    print(f"\n=== {name}: share of points with a partner ===")
    print(d.groupby(pd.cut(d["vo_1e5"], [1, 1.5, 2, 3, 5, 100]), observed=True)["paired"].agg(["size", "mean"]).round(2))
    print(d.groupby(pd.cut(d["lat"], [0, 15, 30, 50, 70, 80, 90]), observed=True)["paired"].agg(["size", "mean"]).round(2))
    print(d.groupby(pd.cut(d["track_len"], [2, 5, 10, 20, 400]), observed=True)["paired"].agg(["size", "mean"]).round(2))
    print("nearest partner of unpaired points (km):")
    print(d.loc[~d["paired"], "nearest_km"].describe(percentiles=[0.25, 0.5, 0.75]).round(0))

print("\nlongest tracks — his:", dv.groupby("track_id").size().nlargest(5).tolist(),
      " yours:", dm.groupby("track_id").size().nlargest(5).tolist())


=== his: share of points with a partner ===
              size  mean
vo_1e5                  
(1.0, 1.5]    3317  0.91
(1.5, 2.0]    3654  0.89
(2.0, 3.0]    4311  0.94
(3.0, 5.0]    3967  0.98
(5.0, 100.0]  2490  0.99
          size  mean
lat                 
(0, 15]   4932  0.82
(15, 30]  2919  0.99
(30, 50]  4559  0.99
(50, 70]  3818  0.99
(70, 80]  1125  0.98
(80, 90]   386  0.84
           size  mean
track_len            
(2, 5]     2636  0.89
(5, 10]    3757  0.93
(10, 20]   5329  0.95
(20, 400]  6017  0.96
nearest partner of unpaired points (km):
count    1060.0
mean     1341.0
std       809.0
min       100.0
25%       935.0
50%      1297.0
75%      1744.0
max      4781.0
Name: nearest_km, dtype: float64

=== yours: share of points with a partner ===
              size  mean
vo_1e5                  
(1.0, 1.5]    3760  0.80
(1.5, 2.0]    3419  0.95
(2.0, 3.0]    4220  0.96
(3.0, 5.0]    4020  0.97
(5.0, 100.0]  2521  0.98
          size  mean
lat                 
(0, 15]   4314